# Turning Headlines into Signals: Using Structured News Metadata to Understand Energy Market Disruptions

Every day, LSEG delivers thousands of Reuters News stories touching the energy sector — pipeline outages, refinery fires, OPEC production adjustments, conflict, sanctions, weather events, regulatory changes. For an analyst trying to build a systematic view of energy supply disruptions, the challenge is not volume; it is **relevance**.

A keyword search for *"oil supply disruption"* will return some useful stories — and a great deal of noise. This notebook demonstrates a different approach: using the **structured classification metadata** that Reuters News attaches to every story to define a precise, repeatable signal for energy supply disruptions.

The goal is not prediction. It is **signal construction** — turning an unstructured information flow into something that can be measured, filtered, and combined with other market data.

## Setup

This notebook uses the [LSEG Data Library for Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python) to retrieve news headlines and story content.

It also relies on a locally included helper package called `newsmetadata`, which provides a simple interface for interpreting the structured classification codes (QCodes) attached to news stories. Rather than working with raw code identifiers, this package resolves them into descriptive labels, navigates parent-child relationships, and extracts the metadata embedded within individual stories. It is used throughout this article to make the classification system readable and explorable.

The internals of `newsmetadata` are beyond the scope of this article — the source is included alongside the notebook for readers who wish to review the implementation details at their own convenience.

In [1]:
import lseg.data as ld
from lseg.data.content import news
from newsmetadata.metadata import metadata

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")

pd.set_option("display.max_colwidth", 120)

In [3]:
# Establish communication with our LSEG data environment.
# Initialize the session - Desktop mode (Workspace)
ld.open_session()

<lseg.data.session.Definition object at 0x1135f63bfe0 {name='workspace'}>

## Signal Definition

Before exploring the data, we define the structured codes that make up our energy supply disruption signal. Each code represents a classification dimension assigned to Reuters News stories — not a keyword, but a **semantic label** applied at publication time.

Centralizing these definitions here makes the signal explicit, easy to modify, and reusable across all queries in this notebook.

In [4]:
# Signal Definition
# These codes define the structured filter for energy supply disruptions.
# Each constant is a list so you can add or remove codes to adjust the signal.
ENERGY_SECTOR = ["B:219"]                   # Energy (TRBC level 1) — covers fossil fuels, renewables, uranium
SUPPLY_TOPICS = ["I:6K", "M:3D0", "M:2Z9"]  # Commodity Production Data; Supply Chain Issues; FCA - Drilling / Production Report
MIDDLE_EAST   = ["G:Q"]                     # Middle East geographic region

# Build reusable query fragments from the signal definition
sector_filter = " OR ".join(ENERGY_SECTOR)
topic_filter  = " OR ".join(SUPPLY_TOPICS)
signal_query  = f"({sector_filter}) AND ({topic_filter})"

print(f"Signal query: {signal_query}")

Signal query: (B:219) AND (I:6K OR M:3D0 OR M:2Z9)


## 1. The Problem with Keywords

The most intuitive way to find news about energy supply disruptions is to search for them by keyword. Let's try that first and see what comes back.

In [5]:
# A broad keyword search for energy supply disruption stories
keyword_results = ld.news.get_headlines(
    query='"oil" AND ("supply disruption" OR "outage" OR "shut down" OR "force majeure")',
    count=50
)

keyword_results.head(15)

,headline,storyId,sourceCode
versionCreated,,,
2026-04-28 10:52:29.000,@staunovo: Oil deliveries from Saudi Aramco to Orlen are on schedule and in line with contracted volumes despite su...,urn:newsml:social:20260428:nTWT6y68qX:1,NS:X
2026-04-28 07:16:23.000,"@VandanaHari_SG: Retweeted @Amena__Bakr: When sourcing oil supplies, China doesn’t care about US sanctions. The US b...",urn:newsml:social:20260428:nTWT3dZwrY:1,NS:X
2026-04-28 06:46:18.449,Scott Bessent Says Iran's 'Creaking' Oil Industry Is Starting To Shut Down Production Due To US Blockade: 'Pumping W...,urn:link:webnews:20260428:nNRA0c5mzh:0,NS:FINGEV
2026-04-28 06:40:19.000,"@staunovo: Retweeted @Amena__Bakr: When sourcing oil supplies, China doesn’t care about US sanctions. The US blockad...",urn:newsml:social:20260428:nTWT2NWlr6:1,NS:X
2026-04-28 06:11:55.154,Scott Bessent Says Iran's 'Creaking' Oil Industry Is Starting To Shut Down Production Due To US Blockade: 'Pumping W...,urn:newsml:newsroom:20260428:nNRA0c56jj:0,NS:BENZIN
2026-04-28 06:09:43.000,"@Amena__Bakr: When sourcing oil supplies, China doesn’t care about US sanctions. The US blockade on Iranian oil ship...",urn:newsml:social:20260428:nTWT5mpjc9:1,NS:X
2026-04-28 02:40:38.000,@Ajay_Bagga: US Treasury Secretary says Iran is starting to shut down oil production. https://t.co/8YCUi0vDwD,urn:newsml:social:20260428:nTWT3PJxLB:1,NS:X
2026-04-28 01:04:54.000,@HFI_Research: Goldman’s report today is getting flack because of this chart. \n\nDon’t blame the analyst. No one wi...,urn:newsml:social:20260428:nTWT51bvbZ:1,NS:X
2026-04-27 21:09:51.000,"@IlliniProgrammr: Replying to @shipwreckedcrew: In the current market, down 0.4% is the new cratering. \n\nStocks a...",urn:newsml:social:20260427:nTWT3TVTTF:1,NS:X


In [6]:
# Extract the source type from the storyId URN structure
# Format: urn:{scheme}:{source}:{date}:{id}:{version}
keyword_results['source'] = keyword_results['storyId'].str.split(':').str[2]

source_counts = keyword_results['source'].value_counts()
print("Story sources in keyword results:")
print(source_counts.to_string())
print(f"\nSocial media stories: {source_counts.get('social', 0)} of {len(keyword_results)} ({source_counts.get('social', 0) / len(keyword_results) * 100:.0f}%)")

Story sources in keyword results:
source
social         25
webnews        15
newsroom        8
reuters.com     2

Social media stories: 25 of 50 (50%)


At first glance, many of these headlines look relevant. But look closer — the `storyId` URN itself reveals important context. Its structure — `urn:{scheme}:{source}:{date}:{id}:{version}` — encodes the **source type** of each story. Parsing it shows that a significant portion of keyword results come from `social` sources (tweets, retweets, commentary on X/Twitter) rather than editorial news content.

This is a key insight: keyword searches don't just match irrelevant *topics* — they pull in entirely different *content types*. Social media posts that happen to contain the right words are returned alongside editorial reporting, with no built-in way to distinguish them. The breakdown above quantifies this noise.

Beyond social media, keyword search may also pull in:

- **Commentary and opinion** pieces that mention disruptions in passing
- **Market roundups** where supply is one of several themes discussed
- **Downstream stories** (demand-side, retail pricing) that happen to use supply-related language
- **Unrelated sectors** where similar vocabulary appears (e.g., power grid outages, supply chain logistics)

The fundamental issue is that **keywords match language, not meaning**. A story can use the word *"disruption"* without being about a supply disruption event. And a story about a pipeline explosion may never use the word *"disruption"* at all.

To build a reliable signal, we need something that captures **what a story is about** — not just what words it contains.

## 2. Defining the Signal with Structured Metadata

Every Reuters News story is classified at publication time with a set of structured codes. These codes describe the story's **topic**, **industry sector**, **named entities**, **geography**, and more. They are assigned by a combination of editorial rules and automated classification — not by keyword matching.

This means we can define our signal not as a bag of words, but as a **combination of structured attributes** that together describe what we care about: energy supply disruptions.

Let's start by identifying the relevant building blocks.

In [7]:
# What does the Energy sector look like in the classification system?
energy_sector = metadata.nodes(ENERGY_SECTOR)
energy_sector

,id,description,label,group,readable,searchable,childrenCount,rcs_code,status,error_message
0,B:219,"Explorers, refiners, marketers and distributors of fossil fuels, uranium and renewable energy, manufacturers of ener...",Energy (TRBC level 1),BusinessSectors,Topic:ENER,True,63,B:219,ok,None


In [8]:
# What sub-sectors exist under Energy?
energy_children = pd.concat([metadata.children(code) for code in ENERGY_SECTOR], ignore_index=True)
energy_children

,id,description,label,group,readable,searchable,childrenCount
0,B:2,"Explorers, refiners, marketers and distributors of fossil fuels, as well as manufacturers of energy-related equipmen...",Energy - Fossil Fuels (TRBC level 2),BusinessSectors,Topic:ENFF,True,38
1,B:220,"Manufacturers of renewable energy equipment, as well as service providers, and producers and distributors of renewab...",Renewable Energy (TRBC level 2),BusinessSectors,B:220,False,20
2,B:224,"Extraction and primary processing of uranium. Includes companies engaged in mining thorium, polonium, carnotite, rad...",Uranium (TRBC level 2),BusinessSectors,B:224,False,4


The classification system organizes energy into meaningful sub-sectors — oil & gas, coal, renewables, nuclear, and so on. Rather than searching for the word *"oil,"* we can target stories that have been **classified** as being about specific energy sub-sectors.

But sector alone is not enough. We also need to narrow by **topic** — specifically, supply-side disruption events.

In [9]:
# Identify supply-related topic codes
supply_topics = metadata.nodes(SUPPLY_TOPICS)
supply_topics

,id,description,label,group,readable,searchable,childrenCount,rcs_code,status,error_message
0,M:3D0,"Unexpected events or conditions that interrupt the normal flow of goods, services, or information within a supply ch...",Supply Chain Issues,MoreTopics,Topic:SUPPIS,True,0,M:3D0,ok,None
1,M:2Z9,"FCA - Report given by mineral, oil and natural gas companies.",FCA - Drilling / Production Report,MoreTopics,Topic:FCDRL,True,0,M:2Z9,ok,None
2,I:6K,Supply data for a particular commodity.,Commodity Production Data,MoreTopics,Topic:COMPRO,True,0,I:6K,ok,None


Now we have two dimensions of classification:

| Dimension | Purpose | Example Codes |
|-----------|---------|---------------|
| **Sector** | *What industry is the story about?* | `ENERGY_SECTOR` — Energy and children |
| **Topic** | *What kind of event is described?* | `SUPPLY_TOPICS` — Supply/Demand, Output/Production |

By requiring **both** dimensions to be present, we can construct a much more precise filter than any keyword search can achieve. A story must be *classified as energy-sector* **and** *classified as covering a supply-side event* to pass through.

## 3. Retrieving the Signal

With our signal defined as an intersection of sector and topic classifications, we can now query for stories that match.

In [10]:
# Retrieve stories matching our signal definition
signal_results = ld.news.get_headlines(
    query=signal_query,
    count=50
)

print(f"Stories matching structured signal: {len(signal_results)}")
signal_results.head(15)

Stories matching structured signal: 50


,headline,storyId,sourceCode
versionCreated,,,
2026-04-28 13:02:09.000,Uganda to offer new oil exploration licensing round in 2026/27 financial year,urn:newsml:reuters.com:20260428:nL8N41B1RP:1,NS:RTRS
2026-04-28 12:30:00.000,Global Instability Accelerates Push for Domestic Energy Security,urn:newsml:reuters.com:20260428:nDjc8LBxhF:1,NS:DJCP
2026-04-28 12:18:13.000,Ameresco brings $23 million MCPS energy savings contract solar projects online,urn:newsml:reuters.com:20260428:nNDL6zfKw5:1,NS:PUBT
2026-04-28 12:18:07.740,Ameresco Partners with Montgomery County Public Schools to Bring Solar Energy Online,urn:link:webnews:20260428:nNRA0cakfc:0,NS:BARCHA
2026-04-28 12:17:19.338,Ameresco Partners with Montgomery County Public Schools to Bring Solar Energy Online,urn:link:webnews:20260428:nNRA0caj52:0,NS:YAHUKI
2026-04-28 12:12:00.000,Press Release: Ameresco Partners with Montgomery County Public Schools to Bring Solar Energy Online,urn:newsml:reuters.com:20260428:nDjc90k8Jz:1,NS:DJCP
2026-04-28 12:05:00.000,Ameresco Partners with Montgomery County Public Schools to Bring Solar Energy Online,urn:newsml:reuters.com:20260428:nDjc2P1HfB:1,NS:DJCP
2026-04-28 12:00:11.179,Croatian regulator approves grid connection fees in boost for renewables,urn:newsml:reuters.com:20260428:nSEEFPsB5a:1,NS:SEE
2026-04-28 12:00:01.000,"Brazil's sugar output set to dip in 2026/27 as ethanol gains, Conab says",urn:newsml:reuters.com:20260428:nS0N40B01Z:2,NS:RTRS


In [11]:
# Apply the same URN source analysis to the structured signal results
signal_results['source'] = signal_results['storyId'].str.split(':').str[2]

signal_source_counts = signal_results['source'].value_counts()
print("Story sources in structured signal results:")
print(signal_source_counts.to_string())
print(f"\nSocial media stories: {signal_source_counts.get('social', 0)} of {len(signal_results)} ({signal_source_counts.get('social', 0) / len(signal_results) * 100:.0f}%)")

Story sources in structured signal results:
source
reuters.com    30
webnews        10
newsroom        6
social          4

Social media stories: 4 of 50 (8%)


Compare these source breakdowns with the keyword results in Section 1. The difference is striking: keyword search returned a mix of `social`, `webnews`, and `newsroom` content — with social media making up a large share. The structured signal, by contrast, is dominated by editorial sources. Social media posts simply don't carry the classification codes that the structured query requires, so they're filtered out by design.

This single comparison — parsing the `storyId` URN from both approaches — tells most of the story. The structured query doesn't just find better *topics*; it finds a fundamentally different *class of content*. No additional metadata inspection is needed to see that.

## 4. What's Inside a Story: Examining the Metadata

To understand why this approach works, it helps to look at the classification attached to an individual story. Each story carries a `subject` array of structured codes — the editorial classification that describes sector, topic, geography, and more. We can extract those raw codes and resolve them into human-readable descriptions using our metadata helper.

> **Note:** Not every story will have subject codes populated at the time you retrieve it. Breaking or high-priority stories are often published immediately and may initially arrive with an empty `subject_codes` array. Metadata such as subject classifications can be added or updated after the initial publication, at which point the story may be re-published with the enriched data. If the story you select has no subject codes, simply try a different index (e.g., `iloc[1]` instead of `iloc[0]`) or choose another story ID from the results.

In [12]:
# Pick a story from our signal results
sample_story_id = signal_results.iloc[1]['storyId']

# Retrieve the story and extract its subject codes
story = news.story.Definition(sample_story_id).get_data()
s = story.data.story
subject_codes = s.subject_codes

# Show some of the raw codes to demonstrate what is available...
print(", ".join(subject_codes[:10]) + ", ...")


B:1004, B:2, B:219, B:5, B:7, E:1, E:E, G:4, G:9, M:1QT, ...


In [13]:
sample_story_id

'urn:newsml:reuters.com:20260428:nDjc8LBxhF:1'

In [14]:
# Resolve the raw codes into readable descriptions
metadata.nodes(subject_codes)

,id,description,label,group,readable,searchable,childrenCount,rcs_code,status,error_message
0,B:2,"Explorers, refiners, marketers and distributors of fossil fuels, as well as manufacturers of energy-related equipmen...",Energy - Fossil Fuels (TRBC level 2),BusinessSectors,Topic:ENFF,True,38.0,B:2,ok,None
1,B:219,"Explorers, refiners, marketers and distributors of fossil fuels, uranium and renewable energy, manufacturers of ener...",Energy (TRBC level 1),BusinessSectors,Topic:ENER,True,63.0,B:219,ok,None
2,B:1004,Companies engaged in the exploration and extraction of crude petroleum and natural gas.,Oil & Gas Exploration and Production (NEC) (TRBC level 5),BusinessSectors,Topic:EXPRO1,True,0.0,B:1004,ok,None
3,B:5,"Producers, refiners and transporters of raw and refined oil and gas products.",Oil & Gas (TRBC level 3),BusinessSectors,Topic:OILG,True,13.0,B:5,ok,None
4,B:7,"Explorers, extractors and producers of crude petroleum and natural gas. Includes operators that recover butane, etha...",Oil & Gas Exploration and Production (TRBC level 4),BusinessSectors,Topic:EXPRO,True,6.0,B:7,ok,None
5,G:4,NaN,Americas,Geography,Topic:AMERS,True,111.0,G:4,ok,None
6,M:1QT,Issues relating to a company's ability to deliver on stated goals.,Execution Risk,MoreTopics,Topic:EXRSKS,True,0.0,M:1QT,ok,None
7,G:9,NaN,North America,Geography,Topic:NAMER,True,57.0,G:9,ok,None
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M:1Z6,error,"[(404, News code identifier not found)]"
9,E:E,"Management issues including internal controls, top management pay, bonuses and corporate governance. Also includes c...",Management Issues / Policies,MoreTopics,Topic:MNGISS,True,31.0,E:E,ok,None


You'll notice that some rows in the table above show a status of `error` with the message *"News code identifier not found"*. These are consistently codes that begin with `P:` — for example, `P:5035454599` or `P:4297904246`. These are not classification codes at all; they are **PermIDs** (Permanent Identifiers), LSEG's system for uniquely identifying real-world entities such as companies, people, and organizations.

PermIDs appear in the `subject` array because it serves as a unified tagging container — it carries not only the editorial classification (topic, sector, geography) but also **entity annotations** that link the story to specific real-world entities. This is by design: it allows consumers of the data to connect a story to the broader LSEG entity ecosystem, where a PermID can be used to look up company fundamentals, ownership structures, or other reference data. Because our metadata helper only resolves news classification codes, PermIDs fall outside its scope and can be safely ignored here.

In [15]:
# For context, here is the story behind those codes
s.content.text

'AUSTIN, Texas, April 28, 2026 (GLOBE NEWSWIRE) -- EnergyWireNews: Global\nenergy markets are once again confronting a familiar vulnerability. Rising\ninstability in the Middle East and renewed concerns surrounding the Strait of\nHormuz, a shipping corridor that handles roughly 20% of global petroleum\nliquids consumption, have intensified calls for the United States and Europe\nto strengthen long-term energy independence. As governments and industries\nreassess the risks tied to overseas supply disruptions, attention is\nincreasingly turning toward politically aligned regions capable of delivering\nsubstantial new energy resources. Against this backdrop, Greenland Energy\nCompany (NASDAQ: GLND) (profile) is advancing an Arctic exploration strategy\ncentered on Greenland\'s Jameson Land Basin where, after drilling two targeted\nwells later this year, the company anticipates securing rights to 70% of the\nbasin and its estimated potential of up to 13 billion barrels of oil. The\ncompany

The raw codes on their own are opaque. But once resolved through the metadata API, they become a structured profile of the story: which sector it covers, what topics it addresses, which geographies are relevant, and more.

This is the key insight: every story carries a machine-readable description of **what it is about**, not just the words it contains. That depth of classification is what allows us to build precise filters. And because it's applied consistently across all stories, the signal is **repeatable** — the same query run tomorrow will apply the same logic to new stories.

The `newsmetadata` helper also provides a convenience method — `metadata.story_nodes(story_id)` — that wraps the extraction and resolution steps above into a single call. We'll use it later when processing multiple stories at once.

## 5. Sharpening the Signal: Adding Geographic Focus

Our current signal captures energy supply stories globally. Depending on the use case, we may want to narrow further — for example, focusing on supply disruptions in a specific producing region.

Structured metadata makes this straightforward. We simply add a geographic dimension to our filter.

In [16]:
# Look up the geographic code for our regional focus
geo_codes = metadata.nodes(MIDDLE_EAST)
geo_codes

,id,label,group,readable,searchable,childrenCount,rcs_code,status,error_message
0,G:Q,Middle East,Geography,Topic:MEAST,True,18,G:Q,ok,None


In [17]:
# Narrow the signal to Middle East energy supply disruptions
geo_filter = " OR ".join(MIDDLE_EAST)
regional_query = f"{signal_query} AND ({geo_filter})"

regional_results = ld.news.get_headlines(
    query=regional_query,
    count=50
)

print(f"Middle East energy supply stories: {len(regional_results)}")
regional_results.head(10)

Middle East energy supply stories: 50


,headline,storyId,sourceCode
versionCreated,,,
2026-04-28 09:54:35.000,"Middle East Crude-Dubai, Murban premiums rebounds as supply concern lingers",urn:newsml:reuters.com:20260428:nL1N41B0DY:1,NS:RTRS
2026-04-28 09:45:10.000,"CORRECTED-Middle East conflict could cause 120 bcm of LNG supply loss from 2026-2030, IEA says",urn:newsml:reuters.com:20260424:nL6N4170HN:1,NS:RTRS
2026-04-28 09:31:11.651,Saudi Arabia's Top 100 Brands Reach $131.9 Billion as Saudi Vision 2030 Drives Diversification Momentum,urn:link:webnews:20260428:nNRA0c7y8p:0,NS:YAHNEX
2026-04-28 08:29:02.000,Nigeria caps jet fuel prices to avert airline disruptions,urn:newsml:reuters.com:20260428:nL6N41B0N4:3,NS:RTRS
2026-04-27 15:21:11.284,Newscasts - Renewables in vogue as Iran war drives up Europe power prices,urn:newsml:reuters.com:20260427:nRTV4DxbDD:12,NS:RTRS
2026-04-27 15:20:00.182,Jordan Petroleum Refinery posts JD 75.5m profit; approves 50% dividends,urn:newsml:newsroom:20260427:nNRA0bvbod:0,NS:ARASER
2026-04-27 15:09:01.333,Newscasts - Renewables in vogue as Iran war drives up Europe power prices,urn:newsml:reuters.com:20260427:nRTV5GxkXQ:10,NS:RTRS
2026-04-27 12:59:59.950,Jordan Petroleum Refinery posts JD 75.5m profit; approves 50% dividends,urn:newsml:newsroom:20260427:nNRA0bsw55:0,NS:MENFOC
2026-04-27 12:49:59.202,Jordan Petroleum Refinery posts JD 75.5m profit; approves 50% dividends,urn:newsml:newsroom:20260427:nNRA0bstly:0,NS:ARAREL


Each additional metadata dimension we layer in **increases precision** without sacrificing the systematic nature of the approach. We could further refine by:

- **Sub-sector**: Narrow to crude oil vs. natural gas vs. LNG
- **Event type**: Distinguish planned maintenance from unplanned outages
- **Named entities**: Focus on specific companies or facilities

The key insight is that these refinements are **compositional** — you build them by combining structured codes, not by inventing more complex keyword patterns.

## 6. Confirming the Difference Over a Common Window

The source breakdowns in Sections 1 and 3 already reveal the core difference: keyword search mixes social media, web aggregation, and editorial content indiscriminately, while the structured signal is inherently filtered to classified, editorial stories. To confirm this holds consistently, we can run both approaches over the **same one-month window** and compare their source composition side by side.

This also lets us check for **blind spots** — stories the structured signal captures that keyword search misses entirely, because the relevant events were described in language the keyword pattern doesn't anticipate.

In [18]:
# Retrieve both sets over the same one-month window
today = pd.Timestamp.now().normalize()
date_from = (today - pd.DateOffset(months=1)).strftime("%Y-%m-%d")
date_to = today.strftime("%Y-%m-%d")

keyword_month = ld.news.get_headlines(
    query='"oil" AND ("supply disruption" OR "outage" OR "shut down" OR "force majeure")',
    start=date_from,
    end=date_to,
    count=100
)

structured_month = ld.news.get_headlines(
    query=signal_query,
    start=date_from,
    end=date_to,
    count=100
)

# Compare source composition side by side
keyword_month['source'] = keyword_month['storyId'].str.split(':').str[2]
structured_month['source'] = structured_month['storyId'].str.split(':').str[2]

kw_sources = keyword_month['source'].value_counts()
st_sources = structured_month['source'].value_counts()

comparison = pd.DataFrame({
    'Keyword Search': kw_sources,
    'Structured Signal': st_sources
}).fillna(0).astype(int)

print(f"Source comparison over {date_from} to {date_to}\n")
print(comparison.to_string())
print(f"\nKeyword total: {len(keyword_month)}  |  Structured total: {len(structured_month)}")
print(f"Keyword social: {kw_sources.get('social', 0)} ({kw_sources.get('social', 0) / len(keyword_month) * 100:.0f}%)  |  Structured social: {st_sources.get('social', 0)} ({st_sources.get('social', 0) / len(structured_month) * 100:.0f}%)")

# Check for blind spots — stories structured signal found that keywords missed
keyword_story_ids = set(keyword_month['storyId'])
structured_only = structured_month[~structured_month['storyId'].isin(keyword_story_ids)]

if len(structured_only) > 0:
    print(f"\n--- Structured signal results missed by keywords ({len(structured_only)} of {len(structured_month)}) ---")
    for _, row in structured_only.head(5).iterrows():
        print(f"  • {row['headline']}")

Source comparison over 2026-03-28 to 2026-04-28

             Keyword Search  Structured Signal
source                                        
newsroom                 11                 22
reuters.com               5                 59
social                   41                  1
webnews                  43                 18

Keyword total: 100  |  Structured total: 100
Keyword social: 41 (41%)  |  Structured social: 1 (1%)

--- Structured signal results missed by keywords (100 of 100) ---
  • DJ Whitehaven Coal Output Down On-Quarter Amid Wet Weather
  • UPDATE 2-Eni-Repsol joint venture seeking to increase gas output at Venezuela's Cardon IV
  • Eni-Repsol joint venture seeking to increase gas output at Venezuela's Cardon IV
  • Chaberton Energy RFP Seeks Farming Partners for two Maryland Agrivoltaics Projects
  • Eni-Repsol joint venture seeking to increase gas output at Venezuela's Cardon IV


The side-by-side source comparison confirms what Sections 1 and 3 already suggested: keyword search pulls in a large share of social media and aggregated web content, while the structured signal is composed almost entirely of editorially classified stories. The blind spots list further illustrates that keywords miss genuine supply events when the story uses different vocabulary.

This is the core value of the structured approach: **it filters by meaning, not language** — and in doing so, it eliminates entire categories of noise (social media, opinion, off-topic mentions) that keyword search cannot distinguish from relevant editorial reporting.

## 7. Measuring the Signal Over Time

Once we have a clean, well-defined signal, we can measure it. A simple but useful metric is **headline frequency** — how many disruption-related headlines are being published per day? Spikes in frequency often correspond to real-world events.

In [41]:
# Retrieve a broader window for time-series analysis
today = pd.Timestamp.now().normalize()
three_months_ago = (today - pd.DateOffset(months=3)).strftime("%Y-%m-%d")
today_str = today.strftime("%Y-%m-%d")

signal_extended = ld.news.get_headlines(
    query=regional_query,
    start=three_months_ago,
    end=today_str,
    count=2500
)

# Build a daily frequency series
signal_extended['date'] = pd.to_datetime(signal_extended.index).normalize().tz_localize(None)

daily_counts = signal_extended.groupby('date').size().reset_index(name='headline_count')

# Reindex only over the range actually covered by the returned headlines.
# The API returns the most recent N headlines (count=500), so earlier dates in the
# requested window may have no coverage — filling those with zero would be misleading.
actual_start = signal_extended['date'].min()
actual_end = signal_extended['date'].max()
daily_counts = daily_counts.set_index('date').reindex(
    pd.date_range(actual_start, actual_end), fill_value=0
).rename_axis('date').reset_index()

print(f"Headlines returned: {len(signal_extended)}")
print(f"Coverage: {actual_start.strftime('%Y-%m-%d')} to {actual_end.strftime('%Y-%m-%d')}")
daily_counts.head(10)

Headlines returned: 2377
Coverage: 2026-01-28 to 2026-04-27


,date,headline_count
0,2026-01-28,18
1,2026-01-29,3
2,2026-01-30,15
3,2026-01-31,2
4,2026-02-01,13
5,2026-02-02,26
6,2026-02-03,37
7,2026-02-04,8
8,2026-02-05,1
9,2026-02-06,32


In [42]:
# Add a rolling average to smooth daily volatility
daily_counts['rolling_7d'] = daily_counts['headline_count'].rolling(window=7, min_periods=1).mean()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=daily_counts['date'],
    y=daily_counts['headline_count'],
    name='Daily Headline Count',
    marker_color='lightsteelblue',
    opacity=0.6
))

fig.add_trace(go.Scatter(
    x=daily_counts['date'],
    y=daily_counts['rolling_7d'],
    name='7-Day Rolling Average',
    line=dict(color='#636EFA', width=2.5)
))

fig.update_layout(
    title='Energy Supply Disruption Signal: Daily Headline Frequency',
    xaxis_title='Date',
    yaxis_title='Number of Headlines',
    template='plotly_white',
    legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99)
)

fig.show()

Spikes in the signal typically correspond to real-world disruption events — a refinery outage, sanctions escalation, weather-driven supply interruptions, or OPEC production decisions. The structured approach ensures these spikes reflect **actual supply events**, not just increased use of disruption-related vocabulary in broader market coverage.

## 8. From Signal to Context: What's Driving the Spike?

The chart above shows daily headline frequency over the coverage window. Some days have noticeably higher counts than others — these are the "spikes." But a spike on its own only tells us *when* disruption coverage intensified, not *why*.

To answer that, we take the **peak day** — the single date with the highest headline count — and decompose its metadata. By examining which classification codes appear most frequently across that day's headlines, we can identify the sub-sectors, geographies, and topics that drove the surge.

In [43]:
# Find the peak day
peak_day = daily_counts.loc[daily_counts['headline_count'].idxmax(), 'date']
print(f"Peak signal day: {peak_day.strftime('%Y-%m-%d')}")

# Retrieve headlines from the peak day
peak_headlines = ld.news.get_headlines(
    query=regional_query,
    start=peak_day.strftime('%Y-%m-%d'),
    end=(peak_day + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
    count=100
)

print(f"\nHeadlines on peak day: {len(peak_headlines)}")
peak_headlines.head(10)

Peak signal day: 2026-03-02

Headlines on peak day: 100


,headline,storyId,sourceCode
versionCreated,,,
2026-03-03 00:00:00.000,RPT-ROI-Oil markets' bet on a brief Iran shock is about to be tested: Bousso,urn:newsml:reuters.com:20260303:nL1N3ZQ0ZK:2,NS:RTRS
2026-03-02 23:18:32.000,《綜述》伊朗報復性襲擊波及能源供應，海灣多國油氣設施停擺,urn:newsml:reuters.com:20260302:nL6T3ZQ1G3:2,NS:RTRS
2026-03-02 23:18:32.000,《综述》伊朗报复性袭击波及能源供应，海湾多国油气设施停摆,urn:newsml:reuters.com:20260302:nL6S3ZQ1G3:2,NS:RTRS
2026-03-02 22:57:08.920,Trio Petroleum (TPET) Rockets As Iran Conflict Tightens Global Oil Supplies,urn:newsml:newsroom:20260302:nNRAzoltab:0,NS:BENZIN
2026-03-02 22:42:36.000,米天然ガス先物が4％上昇、イラン紛争受けたエネルギー供給懸念で,urn:newsml:reuters.com:20260302:nL6N3ZQ1F9:1,NS:RTRS
2026-03-02 22:37:00.287,QatarEnergy Suspends Liquefied Natural Gas Production Following Attack,urn:newsml:newsroom:20260302:nNRAzolkwf:0,NS:AGEMAR
2026-03-02 21:51:57.000,CME Group Inc. - Middle East tensions lifted WTI Crude Oil futures to 8-month high.,urn:newsml:reuters.com:20260302:nNDLbRB1DB:1,NS:PUBT
2026-03-02 21:32:49.000,Açúcar sobe com receio de que guerra no Irã aumente demanda por etanol e reduza produção de açúcar,urn:newsml:reuters.com:20260302:nL6N3ZQ1CY:1,NS:RTRS
2026-03-02 20:46:38.000,ATUALIZA 1-Os futuros do gás natural dos EUA saltam 4% devido às preocupações com o fornecimento de energia decorren...,urn:newsml:reuters.com:20260302:nL1N3ZQ16L:1,NS:RTRS


In [44]:
# Decompose the peak day by examining metadata across headlines
story_ids = peak_headlines["storyId"].head(20).tolist()  # Sample up to 20 headlines

all_metadata = metadata.story_nodes(story_ids)
ok_metadata = all_metadata[all_metadata["status"] == "ok"]

code_counts = ok_metadata["label"].value_counts().head(15)
print("Most frequent metadata labels on peak day:")
print(code_counts.to_string())

Most frequent metadata labels on peak day:
label
Middle East                       32
North America                     27
Europe                            19
Europe, Middle East and Africa    18
Energy (TRBC level 1)             18
Commodities Markets               17
Asia                              17
Emerging Market Countries         17
Gulf                              17
South-West Asia                   17
Asia / Pacific                    17
Energy Markets                    16
Economic Indicators               16
Economic News                     16
Commodity Production Data         16


This metadata decomposition turns a **quantitative spike** into a **qualitative narrative**. Instead of just knowing that supply disruption coverage increased, we can see that it was driven by, for example, Middle East crude oil production stories — which immediately suggests what the market may be reacting to.

Because `metadata.story_nodes()` now accepts a list of story IDs and returns a DataFrame with `story_id`, the bulk decomposition step becomes a single call rather than a manual loop over stories.

## 9. Extending the Approach

Everything demonstrated here for energy supply disruptions is a **pattern**, not a one-off. The same methodology applies to any domain where Reuters News provides structured classification:

- **Geopolitical risk**: Sector + conflict/sanctions topic codes + geography
- **Central bank policy**: Topic codes for monetary policy + named entity codes for specific banks
- **ESG events**: Environmental or governance topic codes + sector + geography
- **M&A activity**: Deal/transaction topic codes + sector + named entities

In each case, the workflow is the same:

1. **Define** the signal as a combination of structured metadata dimensions
2. **Retrieve** stories matching the intersection
3. **Measure** the signal over time
4. **Contextualize** spikes using metadata decomposition
5. **Combine** with other data for deeper analysis

The structured metadata transforms news from a reading experience into an **analytical building block** — something that can be defined precisely, measured consistently, and integrated into broader workflows.

---

**Summary**: Keyword searches match language. Structured metadata matches meaning. By defining a supply disruption signal as an intersection of sector, topic, and geographic classification codes — rather than a pattern of words — we get a cleaner, more complete, and more repeatable view of what's happening in energy markets. That signal can then be measured over time and combined with other market data to support analytical workflows that go well beyond reading headlines.